# RemoteGPU Phase 2 — Colab hardware Vulkan acceptance

This notebook is an **additional** GPU environment, not the sole release baseline. It checks out immutable source commit `16e82dec1eb941e3e79adbdd275368506b2c2b8b`, forces the NVIDIA Vulkan ICD, rejects software Vulkan devices, requires GPU timestamps and independently observed SM activity, renders the reference frame, and requires NVENC H.264 output.

Select a Colab **GPU** runtime before running all cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_URL = 'https://github.com/tysoncaust/Juice-Labs.git'
COMMIT = '16e82dec1eb941e3e79adbdd275368506b2c2b8b'
REPO = Path('/content/Juice-Labs')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', COMMIT], check=True)
print('PINNED_SOURCE_COMMIT=', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import os

EVIDENCE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/RemoteGPU Evidence')
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['RGPU_DRIVE_EVIDENCE_DIR'] = str(EVIDENCE_DIR)
env['RGPU_TIMESTAMP_ITERATIONS'] = '8192'
script = REPO / 'rgpu/renderd/linux/phase2/run_phase2_colab_hardware.sh'
subprocess.run(['bash', str(script)], cwd=REPO, env=env, check=True)


In [ ]:
import json
from IPython.display import JSON, display

out = REPO / 'rgpu/renderd/linux/phase2/out-colab'
summary = (out / 'phase2-colab-summary.txt').read_text()
evidence = json.loads((out / 'phase2-colab-hardware-evidence.json').read_text())
print(summary)
display(JSON(evidence))
print('DRIVE_EVIDENCE_DIR=', EVIDENCE_DIR)
